# CoNR reasoning-trace generation (teacher: Qwen3-Next-80B-A3B-Thinking)

Full-scale run over all of `train.json` (2993 samples). The teacher is given
the context/question **plus the verified gold program and answer**, and asked
to write a natural-language reasoning trace that arrives at that exact program
-- backward rationalization, not independent solving. This guarantees every
trace we keep is grounded in a correct label.

The CoNR prompt/parser/extraction logic here was validated on two 150-sample
trial runs first (see `conr-v1/conr_trace_trial_150.json` and
`conr_trace_trial_150_v2.json`): 92.0% -> 98.0% exact program match after
fixing a nested-parens parser bug, a `<think>`-tag extraction bug, and
widening `max_tokens`. The 3 remaining mismatches in the v2 trial were all
traced to actual errors in `train.json`'s gold labels (a malformed program
missing a closing paren, an invalid `7%` operand, and one gold program whose
division direction contradicts its own question wording) -- not the pipeline.

Run on my own server (2x RTX PRO 6000 96GB + 4 CPU / 16GB RAM). Upload `train.json` via file browser to `/root/data/train.json` before running (or adjust `TRAIN_JSON_PATH` below).

Output: one JSON file with the raw teacher output for all 2993 samples, plus a
validation flag per sample (program exact-match against gold, denylist check).
No mid-run checkpointing -- this runs as a single `llm.generate()` batch call,
same as the trial runs; a failure partway through loses the whole run and
needs a clean re-run from the top.

In [ ]:
%%capture
!pip install -q vllm tabulate
!pip install -q -U "typing_extensions>=4.12" "pydantic>=2.9"
# IMPORTANT: restart the kernel after this cell finishes (Kernel > Restart),
# then re-run from the top. Server's base image ships an older
# typing_extensions that gets imported into the running process before this
# cell runs; upgrading the package on disk doesn't unload the old module
# already in memory, so vllm's `from pydantic import ...` chain still hits
# the stale one (ImportError: cannot import name 'Sentinel' from
# 'typing_extensions') unless the kernel is restarted.

In [ ]:
import json
from pathlib import Path

# Adjust if you uploaded train.json somewhere else via the Server Web UI file browser.
TRAIN_JSON_PATH = Path("/root/valid.json")
OUTPUT_DIR = Path("/root/outputs/conr_trace_full")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not TRAIN_JSON_PATH.exists():
    raise FileNotFoundError(
        f"{TRAIN_JSON_PATH} not found. Upload train.json via the Server Web UI "
        "file browser first, or update TRAIN_JSON_PATH to match where you put it."
    )

train_data = json.load(open(TRAIN_JSON_PATH, encoding="utf-8"))
print(f"Loaded {len(train_data)} samples from {TRAIN_JSON_PATH}")

Loaded 584 samples from /root/valid.json


## Evidence-type classification (for reporting, not sampling)

Classifies each sample by which evidence type its gold program's numeric
arguments come from (table values vs. text values vs. both). Unlike the
150-sample trial, this run uses every sample in `train.json` -- the
classification is kept only so the per-category breakdown can be reported
alongside the overall exact-match rate.

In [3]:
import re

def numbers_in(text):
    return set(re.findall(r"-?\d+\.?\d*", text))

def classify_evidence_type(sample):
    prog = sample["qa"]["program"]
    pre_text = " ".join(sample["pre_text"])
    post_text = " ".join(sample["post_text"])
    text_nums = numbers_in(pre_text) | numbers_in(post_text)
    table_flat = " ".join(" ".join(row) for row in sample["table"])
    table_nums = numbers_in(table_flat)
    prog_nums = set(re.findall(r"-?\d+\.?\d*", prog))

    uses_text = bool(prog_nums & text_nums)
    uses_table = bool(prog_nums & table_nums)

    if uses_table and not uses_text:
        return "table_only"
    elif uses_text and not uses_table:
        return "text_only"
    elif uses_text and uses_table:
        return "table_text"
    return "unclassified"

buckets = {"table_only": [], "table_text": [], "text_only": [], "unclassified": []}
for i, s in enumerate(train_data):
    buckets[classify_evidence_type(s)].append(i)

for k, v in buckets.items():
    print(f"{k}: {len(v)}")

# Use every sample -- no sampling/stratification this time.
sample_idx = list(range(len(train_data)))
print(f"\nTotal samples to process: {len(sample_idx)}")

table_only: 286
table_text: 141
text_only: 54
unclassified: 103

Total samples to process: 584


## Context formatting (same as the 0-shot/1-shot/few-shot/SFT notebooks)

In [4]:
from tabulate import tabulate

def formatting_pre_text(sample):
    return "\n".join(sample["pre_text"])

def formatting_table(sample):
    return tabulate(sample["table"][1:], headers=sample["table"][0], tablefmt="github")

def formatting_post_text(sample):
    return "\n".join(sample["post_text"])

samples = []
for i in sample_idx:
    s = train_data[i]
    samples.append({
        "train_index": i,
        "evidence_type": classify_evidence_type(s),
        "pre_text": formatting_pre_text(s),
        "table": formatting_table(s),
        "post_text": formatting_post_text(s),
        "question": s["qa"]["question"],
        "program": s["qa"]["program"],
        "answer": s["qa"]["exe_ans"],
    })

print(f"Prepared {len(samples)} samples for trace generation.")
samples[0]

Prepared 584 samples for trace generation.


{'train_index': 0,
 'evidence_type': 'text_only',
 'pre_text': 'vay nợ, phải thu và hàng tồn kho tiếp tục tăng cao: phải thu tăng mạnh gần 50%  lên mức 8.344 tỷ đồng, trong đó chủ yếu là phải thu ngắn hạn khác (5.925 tỷ đồng).\nhàng tồn kho cũng tăng mạnh 15% so với đầu năm lên 5.310 tỷ đồng với 18 dự án bất  động sản dở dang.\ntổng nợ ngắn hạn và dài hạn của dxg cũng tăng lên 3.401 tỷ đồng  (tăng 16% so với năm trước).\ndòng tiền kinh doanh 3 quý vừa qua âm 100 tỷ: hai năm trước dxg cũng liên tục bị  âm dòng tiền khoảng 1.000 tỷ mỗi năm.\ndo đó để bổ sung vốn lưu động và đầu tư dự  án, dxg phải tiếp tục huy động vốn thông qua phát hành trái phiếu.\ntính đến hết quý  3, tổng giá trị trái phiếu của dxg đạt 2.102 tỷ đồng (tăng gần 4 lần so với đầu năm),  có kỳ hạn từ 2 – 5 năm.\nđiều này dẫn đến áp lực về chi phí lãi vay lớn.\nvẫn tồn tại vướng mắc đối với các dự án liên quan đến petroland: hai dự án được  chuyển nhượng từ petroland là dự án chung cư thăng long và chuyển nhượng cổ  phần 

## CoNR prompt

Teacher-facing system prompt is distinct from the student's `SYSTEM_MESSAGE`
used elsewhere in this repo: it explicitly reveals the gold program/answer and
asks for a `<think>...</think>` + `<program>...</program>` trace that
rationalizes -- without leaking that it was told the answer -- exactly that
program. See `conr_prompt_design.md` (project notes) for the full rationale.

In [5]:
CONR_SYSTEM_PROMPT = """You are a senior financial analyst writing out your reasoning process for a
numerical question about a Vietnamese financial document. You are given the
context (text before a table, the table itself, text after the table), the
question, and the VERIFIED CORRECT computation program and final answer.

Your task: write a natural-language reasoning trace that a real analyst would
follow to arrive at exactly this program and answer -- as if you were solving
it for the first time, not as if you already knew the answer. Do not mention
that the answer was given to you.

### OUTPUT FORMAT (strict):
<think>
[Your reasoning in Vietnamese, 3-6 sentences. Must:
 1. Identify which specific values are needed and WHERE they come from
    (quote the exact row/column label from the table, or the exact sentence
    from pre_text/post_text) -- never state a number without saying where it
    was found.
 2. Explain WHY each operator was chosen (e.g. "since the question asks for
    a ratio between two periods, we divide the later value by the earlier
    one" rather than just stating the operator name).
 3. If the program has multiple steps, walk through them in the same order
    as the program, referring to intermediate results the way the program
    does (step 1 result, step 2 result, ...).]
</think>
<program>PROGRAM_STRING_EXACTLY_AS_GIVEN</program>

### RULES:
- The <program> block must reproduce the given gold program EXACTLY
  (same operators, same argument order, same #N references). Do not simplify,
  reformat, or "improve" it.
- Never write phrases like "as given", "we are told the answer is", "vi de bai
  cho biet dap an la" -- the trace must read as an independent derivation.
- FORBIDDEN PATTERN -- do not, under any circumstances, refer back to the
  program or answer you were handed as something you were "given" or "told".
  Concretely, never write phrasing like:
    - "chuong trinh duoc cho la ..." / "chương trình được cho là ..."
    - "ket qua duoc cho la ..." / "kết quả được cho là ..."
    - "theo chuong trinh duoc cho" / "theo chương trình được cho"
  If your derivation doesn't independently arrive at the program, silently
  redo the reasoning until it does -- do not narrate the discrepancy or
  reference the program/answer as an external given. (Note: citing what the
  CONTEXT itself states, e.g. "van ban cho biet ty le la 20%" / "văn bản cho
  biết tỷ lệ là 20%", is fine and expected -- that's citing a source document,
  not revealing you were told the final program/answer.)
- If a value must be read off a table row, name that row's label verbatim
  as it appears in the table's first column (or header, if it's a column
  lookup) -- this is what lets us later verify grounding, not just the final
  number.
- Keep the <think> block to 3-6 sentences; do not pad with generic financial
  commentary unrelated to the calculation."""

CONR_USER_FRAME = """### CONTEXT:
[TEXT BEFORE TABLE]
{pre_text}

[TABLE]
{table}

[TEXT AFTER TABLE]
{post_text}

### QUESTION:
{question}

### VERIFIED CORRECT PROGRAM (for you to rationalize, not to reveal you were told):
{program}

### VERIFIED CORRECT ANSWER:
{answer}

### YOUR TASK:
Write the <think>...</think> reasoning trace and <program> block as instructed."""

## Load the teacher model with vLLM

`tensor_parallel_size=2` splits the model across both GPUs. First run will
download the full model from Hugging Face (~160GB in BF16) -- this can take
15-30+ minutes depending on bandwidth, during which the GPUs are idle but still
billed. Consider starting this cell and doing something else while it downloads.

In [6]:
import os

# RTX PRO 6000 (Blackwell, SM120/compute capability 12.0) hits two separate
# FlashInfer JIT-compile failures with this model, at two different stages:
#
# 1. MoE kernels: vLLM's default "auto" moe_backend picks FlashInfer CUTLASS,
#    whose codegen doesn't support SM120 -> "RuntimeError: No supported CUDA
#    architectures found for major versions [12]". Fixed by forcing
#    moe_backend="triton" below (a real vLLM EngineArgs field).
#
# 2. Sampler: vLLM defaults to FlashInfer for top-k/top-p sampling too. Its
#    arch-support check (flashinfer/jit/core.py: check_cuda_arch) doesn't
#    recognize SM120 as valid yet and raises "RuntimeError: FlashInfer
#    requires GPUs with sm75 or higher" -- misleading message (SM120 > SM75),
#    it's really "SM120 isn't in FlashInfer's known-good list yet".
#    VLLM_USE_FLASHINFER_SAMPLER is a real vLLM env var (see vllm/envs.py);
#    setting it to "0" forces the plain PyTorch-native top-k/top-p sampler
#    instead, sidestepping this JIT path entirely. Must be set before vllm
#    is imported (env vars are read at import/engine-init time).
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"

from vllm import LLM, SamplingParams

MODEL_NAME = "Qwen/Qwen3-Next-80B-A3B-Thinking"

llm = LLM(
    model=MODEL_NAME,
    tensor_parallel_size=2,
    dtype="bfloat16",
    max_model_len=16384,   # generous headroom for long <think> traces
    gpu_memory_utilization=0.90,
    trust_remote_code=True,
    moe_backend="triton",
    # Qwen3-Next's hybrid attention uses a separate Mamba cache block per
    # concurrently-running sequence (unlike plain attention KV cache, which
    # can be shared/paged more flexibly). vLLM's default max_num_seqs=1024
    # exceeded what fits in the ~5.4GiB of KV cache memory left over after
    # loading the 74.3GiB/GPU model weights, which only leaves room for 861
    # Mamba cache blocks -- vLLM refuses to start rather than silently running
    # with fewer concurrent sequences than requested:
    #   ValueError: max_num_seqs (1024) exceeds available Mamba cache blocks (861)
    # Capped below the observed 861 limit with some headroom. If you hit this
    # again with a different max_model_len/gpu_memory_utilization (which change
    # how much memory is left for the KV/Mamba cache), lower this further.
    max_num_seqs=800,
)

tokenizer = llm.get_tokenizer()
print(f"Loaded {MODEL_NAME} across 2 GPUs.")

INFO 07-22 07:27:51 [api_utils.py:273] non-default args: {'trust_remote_code': True, 'dtype': 'bfloat16', 'max_model_len': 16384, 'tensor_parallel_size': 2, 'moe_backend': 'triton', 'gpu_memory_utilization': 0.9, 'max_num_seqs': 800, 'disable_log_stats': True, 'model': 'Qwen/Qwen3-Next-80B-A3B-Thinking'}


INFO 07-22 07:27:52 [model.py:619] Resolved architecture: Qwen3NextForCausalLM
INFO 07-22 07:27:52 [model.py:1776] Using max model len 16384
INFO 07-22 07:27:52 [scheduler.py:252] Chunked prefill is enabled with max_num_batched_tokens=16384.
INFO 07-22 07:27:52 [vllm.py:1042] Asynchronous scheduling is enabled.
INFO 07-22 07:27:52 [kernel.py:292] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
(EngineCore pid=5200) INFO 07-22 07:27:57 [core.py:114] Initializing a V1 LLM engine (v0.25.1) with config: model='Qwen/Qwen3-Next-80B-A3B-Thinking', speculative_config=None, tokenizer='Qwen/Qwen3-Next-80B-A3B-Thinking', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=16384, download_dir=None, load_format=auto, tensor_parallel_size=2, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_ba

(Worker pid=5214) (Worker pid=5215) Failed to get device capability: SM 12.x requires CUDA >= 12.9.
Failed to get device capability: SM 12.x requires CUDA >= 12.9.
(Worker pid=5215) (Worker pid=5214) Failed to get device capability: SM 12.x requires CUDA >= 12.9.
Failed to get device capability: SM 12.x requires CUDA >= 12.9.


(Worker pid=5214) INFO 07-22 07:28:03 [pynccl.py:113] vLLM is using nccl==2.28.9
(Worker pid=5215) (Worker pid=5214) WARNING 07-22 07:28:04 [symm_mem.py:66] SymmMemCommunicator: Device capability 12.0 not supported, communicator is not available.
WARNING 07-22 07:28:04 [symm_mem.py:66] SymmMemCommunicator: Device capability 12.0 not supported, communicator is not available.
(Worker pid=5215) (Worker pid=5214) WARNING 07-22 07:28:04 [custom_all_reduce.py:162] Custom allreduce is disabled because your platform lacks GPU P2P capability or P2P test failed. To silence this warning, specify disable_custom_all_reduce=True explicitly.
WARNING 07-22 07:28:04 [custom_all_reduce.py:162] Custom allreduce is disabled because your platform lacks GPU P2P capability or P2P test failed. To silence this warning, specify disable_custom_all_reduce=True explicitly.
(Worker pid=5214) INFO 07-22 07:28:04 [cuda_communicator.py:264] Using ['PYNCCL'] all-reduce backends (in dispatch order) for group 'tp:0' out 

Loading safetensors checkpoint shards:   0% Completed | 0/41 [00:00<?, ?it/s]


(Worker_TP1 pid=5215) INFO 07-22 07:29:59 [interface.py:890] Setting attention block size to 544 tokens to ensure that attention page size is >= mamba page size.
(Worker_TP1 pid=5215) INFO 07-22 07:29:59 [interface.py:914] Padding mamba page size by 1.49% to ensure that mamba page size and attention page size are exactly equal.
(Worker_TP0 pid=5214) INFO 07-22 07:30:01 [default_loader.py:430] Loading weights took 113.02 seconds
(Worker_TP0 pid=5214) INFO 07-22 07:30:01 [unquantized.py:334] Using MoEPrepareAndFinalizeNoDPEPModular
(Worker_TP0 pid=5214) INFO 07-22 07:30:02 [gpu_model_runner.py:5306] Model loading took 74.3 GiB memory and 115.169520 seconds
(Worker_TP0 pid=5214) INFO 07-22 07:30:02 [interface.py:890] Setting attention block size to 544 tokens to ensure that attention page size is >= mamba page size.
(Worker_TP0 pid=5214) INFO 07-22 07:30:02 [interface.py:914] Padding mamba page size by 1.49% to ensure that mamba page size and attention page size are exactly equal.
(Worker

(Worker_TP1 pid=5215) (Worker_TP0 pid=5214) 2026-07-22 07:30:33,072 - INFO - autotuner.py:651 - flashinfer.jit: [Autotuner]: Autotuning process starts ...
2026-07-22 07:30:33,073 - INFO - autotuner.py:651 - flashinfer.jit: [Autotuner]: Autotuning process starts ...
(Worker_TP0 pid=5214) (Worker_TP1 pid=5215) 2026-07-22 07:30:33,296 - INFO - autotuner.py:674 - flashinfer.jit: [Autotuner]: Autotuning process ends
2026-07-22 07:30:33,296 - INFO - autotuner.py:674 - flashinfer.jit: [Autotuner]: Autotuning process ends


(Worker_TP0 pid=5214) (Worker_TP1 pid=5215) INFO 07-22 07:30:33 [cutedsl_warmup.py:97] Skipping CuTeDSL warmup because no compile units were requested.
INFO 07-22 07:30:33 [cutedsl_warmup.py:97] Skipping CuTeDSL warmup because no compile units were requested.


Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|█████████████████| 51/51 [00:24<00:00,  2.06it/s]
Capturing CUDA graphs (decode, FULL): 100%|████████████████████████████████████| 51/51 [00:19<00:00,  2.60it/s]


(Worker_TP1 pid=5215) INFO 07-22 07:31:18 [gpu_worker.py:771] CUDA graph pool memory: 2.52 GiB (actual), 2.23 GiB (estimated), difference: 0.28 GiB (11.2%).
(Worker_TP0 pid=5214) INFO 07-22 07:31:18 [gpu_model_runner.py:6707] Graph capturing finished in 45 secs, took 2.52 GiB
(Worker_TP0 pid=5214) INFO 07-22 07:31:18 [gpu_worker.py:771] CUDA graph pool memory: 2.52 GiB (actual), 2.23 GiB (estimated), difference: 0.28 GiB (11.2%).
(Worker_TP1 pid=5215) INFO 07-22 07:31:18 [jit_monitor.py:73] Kernel JIT monitor activated; monitored JIT compilations during inference will use mode=warn.
(Worker_TP0 pid=5214) INFO 07-22 07:31:18 [jit_monitor.py:73] Kernel JIT monitor activated; monitored JIT compilations during inference will use mode=warn.
(EngineCore pid=5200) INFO 07-22 07:31:19 [core.py:337] init engine (profile, create kv cache, warmup model) took 77.15 s (compilation: 19.81 s)
(EngineCore pid=5200) INFO 07-22 07:31:20 [kernel.py:292] Final IR op priority after setting platform default

## Generate traces (batched via vLLM)

vLLM handles all 2993 requests as a single batch internally (continuous
batching), so this is one `llm.generate()` call rather than a per-sample loop.
This will take considerably longer than the 150-sample trial (~20x the
requests) -- expect this cell to run for a while.

In [7]:
prompts = []
for s in samples:
    user_msg = CONR_USER_FRAME.format(
        pre_text=s["pre_text"], table=s["table"], post_text=s["post_text"],
        question=s["question"], program=s["program"], answer=s["answer"],
    )
    messages = [
        {"role": "system", "content": CONR_SYSTEM_PROMPT},
        {"role": "user", "content": user_msg},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    prompts.append(text)

# max_tokens=4096 (was 2048): in the first 150-sample trial run, 3 samples got
# cut off mid-generation because the model second-guessed itself (re-deriving
# the calculation, sometimes switching to English self-talk) before emitting
# <program>, burning the whole 2048-token budget on that detour. Doubling the
# budget gives it room to finish even when it goes down that path.
sampling_params = SamplingParams(
    temperature=0.6,
    top_p=0.9,
    max_tokens=4096,
)

outputs = llm.generate(prompts, sampling_params)
print(f"Generated {len(outputs)} responses.")

Rendering prompts:   0%|          | 0/584 [00:00<?, ?it/s]

Processed prompts: 100%|█| 584/584 [03:44<00:00,  2.60it/s, est. speed input: 5415.81 toks/s, output: 724.90 to

Generated 584 responses.


## Parse + validate each trace

Two checks per sample, matching the plan in `conr_prompt_design.md`:
1. `<program>` block must EXACT-match the gold program (same operators/args/
   order/#N refs) via the existing `parse_program`/`compute_program_accuracy`
   logic used across the other notebooks -- not just execution-equivalent.
2. `<think>` block must not contain leak phrases suggesting the teacher was
   told the answer.

In [8]:
"""
Parser + evaluation utilities for FinQA/VLSP-2025 style computation programs.
(Same implementation as the 0-shot/1-shot/few-shot/SFT notebooks, with one fix:
parse_program now handles nested parentheses inside arguments, e.g. table column
names like "Bien LN HDSXKD (%)" or "P/E (x)" -- the original regex-based
`([a-zA-Z_]+)\\(([^()]*)\\)` couldn't match across a paren nested inside the
argument list, so any program referencing such a column raised ValueError and
got scored as a false mismatch even when generated == gold verbatim. Found via
manual review of the first 150-sample trial run: 9 of 12 "mismatches" were
actually this bug, not a real generation difference.)
"""

from typing import List, Tuple, Optional

VALID_OPERATORS = {
    "add", "subtract", "multiply", "divide", "exp", "greater",
    "table_sum", "table_average", "table_max", "table_min",
}
_NUM_RE = re.compile(r"^-?\d+(\.\d+)?$")
_REF_RE = re.compile(r"^#(\d+)$")


def _split_top_level(s: str, sep: str = ",") -> List[str]:
    parts, depth, current = [], 0, []
    for ch in s:
        if ch == "(":
            depth += 1; current.append(ch)
        elif ch == ")":
            depth -= 1; current.append(ch)
        elif ch == sep and depth == 0:
            parts.append("".join(current)); current = []
        else:
            current.append(ch)
    if current:
        parts.append("".join(current))
    return [p.strip() for p in parts if p.strip() != ""]


def parse_program(program_str: str) -> List[Tuple[str, List[str]]]:
    program_str = program_str.strip().rstrip(",").strip()
    if not program_str:
        raise ValueError("Empty program string.")

    steps: List[Tuple[str, List[str]]] = []
    name_pattern = re.compile(r"\s*([a-zA-Z_]+)\(")
    pos = 0
    text = program_str
    while pos < len(text):
        m = name_pattern.match(text, pos)
        if not m:
            raise ValueError(f"Malformed program near: '{text[pos:pos+30]}...'")
        op = m.group(1).strip()
        if op not in VALID_OPERATORS:
            raise ValueError(f"Unknown operator: '{op}'")

        # Find the matching close paren by depth-counting, so parens nested
        # inside an argument (e.g. a column name like "ROE (%)") don't
        # prematurely end the call.
        start = m.end() - 1  # index of the opening '('
        depth = 0
        i = start
        while i < len(text):
            if text[i] == "(":
                depth += 1
            elif text[i] == ")":
                depth -= 1
                if depth == 0:
                    break
            i += 1
        else:
            raise ValueError(f"Unbalanced parens near: '{text[start:start+30]}...'")

        args = _split_top_level(text[start + 1:i], sep=",")
        steps.append((op, args))
        pos = i + 1
        if pos < len(text) and text[pos] == ",":
            pos += 1

    if not steps:
        raise ValueError("No valid steps parsed.")
    return steps


def _normalize_program(program_str: str) -> List[Tuple[str, Tuple[str, ...]]]:
    steps = parse_program(program_str)
    normalized = []
    for op, args in steps:
        norm_args = []
        for a in args:
            a = a.strip()
            if _REF_RE.match(a) or a.lower() == "none":
                norm_args.append(a.lower())
            else:
                cleaned = a.replace(",", "").replace("%", "").replace("$", "").strip()
                try:
                    norm_args.append(f"{round(float(cleaned), 6)}")
                except ValueError:
                    norm_args.append(a)
        normalized.append((op, tuple(norm_args)))
    return normalized


def is_exact_program_match(generated_program: str, gold_program: str) -> bool:
    """Exact structural match (not just execution-equivalent) -- if the teacher
    reformatted/simplified the gold program, we want to know and discard it."""
    try:
        return _normalize_program(generated_program) == _normalize_program(gold_program)
    except ValueError:
        return False


# Regex patterns (not plain substrings) so common, legitimate phrasing like
# "van ban cho biet ..." / "văn bản cho biết ..." (citing what the source
# document says) doesn't false-positive. The first 150-sample trial run used
# plain substrings ("cho biết", "được cho") and flagged 6 samples, of which
# only 4 were real leaks -- 2 were the model citing pre_text/table content,
# which is expected and fine. These patterns require "chương trình"/"kết
# quả"/"đáp án" immediately followed by "được cho" to actually catch the
# forbidden pattern (the teacher referring to the program/answer itself as
# something it was handed), not just any use of "cho biết"/"được cho".
LEAK_PATTERNS = [
    r"chương trình\s+được cho",
    r"kết quả\s+được cho",
    r"đáp án\s+được cho",
    r"theo chương trình được cho",
    r"đã biết trước",
    r"\bverified\b",
    r"\bas given\b",
    r"\bwe are told\b",
    r"\bgiven answer\b",
    r"\bgiven program\b",
]


def find_leak_phrases(think_block: str) -> List[str]:
    lowered = think_block.lower()
    return [pat for pat in LEAK_PATTERNS if re.search(pat, lowered)]


def extract_think_and_program(raw_text: str):
    """
    Extract the reasoning trace and program string from the model's raw output.

    Qwen3-Next-Thinking doesn't emit a literal opening `<think>` tag -- the
    chat template already puts the model in "thinking" mode at the start of
    the assistant turn, so the decoded text starts directly with the
    reasoning content and only closes with `</think>`. The first 150-sample
    trial run showed this is 100% consistent (0/150 had an opening tag), which
    made every `think` come back None under the original `<think>(.*?)</think>`
    regex. So: treat everything before the first `</think>` as the reasoning
    block (whether or not an opening tag happens to be present), and extract
    `<program>...</program>` as before.
    """
    close_think_idx = raw_text.find("</think>")
    if close_think_idx == -1:
        think = None
    else:
        think = raw_text[:close_think_idx]
        think = re.sub(r"^\s*<think>\s*", "", think)  # strip an opening tag if present
        think = think.strip()

    program_match = re.search(r"<program>(.*?)</program>", raw_text, re.DOTALL)
    program = program_match.group(1).strip() if program_match else None

    return think, program

In [9]:
results = []
n_exact_match = 0
n_missing_tags = 0
n_leak = 0

for s, output in zip(samples, outputs):
    raw_text = output.outputs[0].text
    think, program = extract_think_and_program(raw_text)

    exact_match = is_exact_program_match(program, s["program"]) if program else False
    leaks = find_leak_phrases(think) if think else []

    if think is None or program is None:
        n_missing_tags += 1
    if exact_match:
        n_exact_match += 1
    if leaks:
        n_leak += 1

    results.append({
        "train_index": s["train_index"],
        "evidence_type": s["evidence_type"],
        "question": s["question"],
        "gold_program": s["program"],
        "gold_answer": s["answer"],
        "raw_output": raw_text,
        "parsed_think": think,
        "parsed_program": program,
        "exact_program_match": exact_match,
        "leak_phrases_found": leaks,
    })

print(f"Total: {len(results)}")
print(f"Exact program match: {n_exact_match} ({100 * n_exact_match / len(results):.1f}%)")
print(f"Missing <think>/<program> tags: {n_missing_tags}")
print(f"Traces with leak phrases: {n_leak}")

Total: 584
Exact program match: 578 (99.0%)
Missing <think>/<program> tags: 5
Traces with leak phrases: 7


## Save all results for review

Saves every sample's raw output + parsed fields + validation flags, regardless
of pass/fail, so the full set can be read through to judge trace quality and
refine `CONR_SYSTEM_PROMPT` before scaling to the full train set.

In [ ]:
output_path = OUTPUT_DIR / "conr_trace_full_valid.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump({
        "model": MODEL_NAME,
        "n_samples": len(results),
        "summary": {
            "exact_program_match": n_exact_match,
            "missing_tags": n_missing_tags,
            "leak_phrases": n_leak,
        },
        "results": results,
    }, f, ensure_ascii=False, indent=2)

print(f"Saved {len(results)} results to {output_path}")
print("Download this file from the Server Web UI file browser.")